# 🛢️ Deep-SAR Oil Spill Detection — Kaggle GPU Training
**Model**: DeepLabV3+ | MobileNetV3-Large | scSE Attention | Sentinel + PALSAR

> **Before running**: Upload your `sos` dataset folder to Kaggle Datasets.
> Update `DATASET_NAME` below to match whatever you named it on Kaggle.

In [ ]:
!pip install albumentations==1.3.1 -q

In [ ]:
import os, numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torchvision.models.segmentation as seg_models
from torch.utils.data import Dataset, DataLoader, random_split, ConcatDataset
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print('Imports OK')

In [ ]:
# ── CONFIG ─────────────────────────────────────────────────────────────────
# Change this to the slug name of your Kaggle Dataset
DATASET_NAME = 'sos-oil-spill'
DATA_ROOT    = f'/kaggle/input/{DATASET_NAME}/sos'

TRAIN_IMG_DIR         = os.path.join(DATA_ROOT, 'train', 'sentinel', 'image')
TRAIN_MASK_DIR        = os.path.join(DATA_ROOT, 'train', 'sentinel', 'label')
TEST_IMG_DIR          = os.path.join(DATA_ROOT, 'test',  'sentinel', 'image')
TEST_MASK_DIR         = os.path.join(DATA_ROOT, 'test',  'sentinel', 'label')
TRAIN_IMG_DIR_PALSAR  = os.path.join(DATA_ROOT, 'train', 'palsar',   'image')
TRAIN_MASK_DIR_PALSAR = os.path.join(DATA_ROOT, 'train', 'palsar',   'label')
TEST_IMG_DIR_PALSAR   = os.path.join(DATA_ROOT, 'test',  'palsar',   'image')
TEST_MASK_DIR_PALSAR  = os.path.join(DATA_ROOT, 'test',  'palsar',   'label')

CHECKPOINT_DIR = '/kaggle/working/checkpoints'
PRED_DIR       = '/kaggle/working/predictions'

IMAGE_SIZE    = 256
BATCH_SIZE    = 8        # GPU can handle 8+
NUM_EPOCHS    = 50
LEARNING_RATE = 1e-4
NUM_WORKERS   = 2
PIN_MEMORY    = True
VAL_SPLIT     = 0.15
NUM_CLASSES   = 1
DEVICE        = 'cuda' if torch.cuda.is_available() else 'cpu'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(PRED_DIR,       exist_ok=True)

print(f'Device     : {DEVICE}')
print(f'Batch Size : {BATCH_SIZE}')
print(f'Epochs     : {NUM_EPOCHS}')
print(f'Train path : {TRAIN_IMG_DIR}')
print(f'Path exists: {os.path.exists(TRAIN_IMG_DIR)}')

In [ ]:
# ── DATASET ─────────────────────────────────────────────────────────────────
def get_transforms(train=True):
    if train:
        return A.Compose([
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.GaussianBlur(p=0.2),
            A.GaussNoise(p=0.2),
            A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.3),
            A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
            ToTensorV2(),
        ])
    return A.Compose([
        A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
        ToTensorV2(),
    ])

class SOSDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None):
        self.img_dir   = img_dir
        self.mask_dir  = mask_dir
        self.transform = transform
        self.images    = sorted([f for f in os.listdir(img_dir)
                                  if f.endswith(('.png', '.jpg', '.tif', '.tiff'))])

    def __len__(self):  return len(self.images)
    def __repr__(self): return f'SOSDataset(n={len(self.images)})'

    def __getitem__(self, idx):
        name      = self.images[idx]
        image     = np.array(Image.open(os.path.join(self.img_dir,  name)).convert('RGB'))
        mask      = np.array(Image.open(os.path.join(self.mask_dir, name)).convert('L'))
        mask      = (mask > 127).astype(np.float32)
        if self.transform:
            aug   = self.transform(image=image, mask=mask)
            image, mask = aug['image'], aug['mask']
        return image, mask.unsqueeze(0)

def get_loaders(batch_size=BATCH_SIZE):
    sent_ds  = SOSDataset(TRAIN_IMG_DIR,        TRAIN_MASK_DIR,        get_transforms(True))
    pals_ds  = SOSDataset(TRAIN_IMG_DIR_PALSAR, TRAIN_MASK_DIR_PALSAR, get_transforms(True))
    full_ds  = ConcatDataset([sent_ds, pals_ds])
    print(f'Combined training set: {len(full_ds)} samples')

    val_size   = int(len(full_ds) * VAL_SPLIT)
    train_size = len(full_ds) - val_size
    train_ds, val_ds = random_split(full_ds, [train_size, val_size])

    # Apply val transforms (no augmentation) to validation split
    val_ds.dataset = ConcatDataset([
        SOSDataset(TRAIN_IMG_DIR,        TRAIN_MASK_DIR,        get_transforms(False)),
        SOSDataset(TRAIN_IMG_DIR_PALSAR, TRAIN_MASK_DIR_PALSAR, get_transforms(False)),
    ])

    test_ds = ConcatDataset([
        SOSDataset(TEST_IMG_DIR,        TEST_MASK_DIR,        get_transforms(False)),
        SOSDataset(TEST_IMG_DIR_PALSAR, TEST_MASK_DIR_PALSAR, get_transforms(False)),
    ])

    kw = dict(batch_size=batch_size, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    return (
        DataLoader(train_ds, shuffle=True,  **kw),
        DataLoader(val_ds,   shuffle=False, **kw),
        DataLoader(test_ds,  shuffle=False, **kw),
    )

print('Dataset ready ✓')

In [ ]:
# ── MODEL ────────────────────────────────────────────────────────────────────
class ChannelSE(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc   = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid())
    def forward(self, x):
        b, c, _, _ = x.shape
        return x * self.fc(self.pool(x).view(b, c)).view(b, c, 1, 1)

class SpatialSE(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, 1, kernel_size=1)
    def forward(self, x):
        return x * torch.sigmoid(self.conv(x))

class scSEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.cse = ChannelSE(channels, reduction)
        self.sse = SpatialSE(channels)
    def forward(self, x): return self.cse(x) + self.sse(x)

class OilSpillDeepLab(nn.Module):
    def __init__(self, num_classes=1):
        super().__init__()
        base = seg_models.deeplabv3_mobilenet_v3_large(weights='DEFAULT')
        self.backbone         = base.backbone
        self.classifier       = base.classifier
        self.classifier[-1]   = nn.Conv2d(256, num_classes, kernel_size=1)
        self.scse             = scSEBlock(channels=960)

    def forward(self, x):
        feats = self.backbone(x)
        high  = feats.get('out', feats.get('high', list(feats.values())[-1]))
        high  = self.scse(high)
        out   = self.classifier(high)
        return nn.functional.interpolate(out, size=x.shape[-2:], mode='bilinear', align_corners=True)

def get_model(device):
    model = OilSpillDeepLab().to(device)
    print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')
    return model

print('Model architecture ready ✓')

In [ ]:
# ── LOSS & METRICS ───────────────────────────────────────────────────────────
def dice_score(pred, target, threshold=0.5, eps=1e-6):
    pred = (torch.sigmoid(pred) > threshold).float()
    i    = (pred * target).sum(dim=(1,2,3))
    u    = pred.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3))
    return ((2.0 * i + eps) / (u + eps)).mean()

def iou_score(pred, target, threshold=0.5, eps=1e-6):
    pred = (torch.sigmoid(pred) > threshold).float()
    i    = (pred * target).sum(dim=(1,2,3))
    u    = pred.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3)) - i
    return ((i + eps) / (u + eps)).mean()

def bce_dice_loss(pred, target, bce_weight=0.5, eps=1e-6):
    # Label smoothing on BCE only
    bce       = nn.functional.binary_cross_entropy_with_logits(pred, target * 0.9 + 0.05)
    pred_soft = torch.sigmoid(pred)
    i         = (pred_soft * target).sum(dim=(1,2,3))
    u         = pred_soft.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3))
    dice      = 1.0 - ((2.0 * i + eps) / (u + eps)).mean()
    return bce_weight * bce + (1.0 - bce_weight) * dice

def save_checkpoint(model, optimizer, epoch, path):
    torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                'optimizer_state': optimizer.state_dict()}, path)

def load_checkpoint(path, model, optimizer=None):
    ckpt = torch.load(path, map_location='cpu')
    model.load_state_dict(ckpt['model_state'])
    if optimizer: optimizer.load_state_dict(ckpt['optimizer_state'])
    return ckpt['epoch']

print('Loss / metrics ready ✓')

In [ ]:
# ── TRAINING FUNCTIONS ───────────────────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    for imgs, masks in tqdm(loader, desc='Train', leave=False):
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        preds = model(imgs)
        if preds.shape != masks.shape:
            preds = nn.functional.interpolate(preds, size=masks.shape[-2:],
                                              mode='bilinear', align_corners=False)
        loss = bce_dice_loss(preds, masks)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, device):
    model.eval()
    dice_total = iou_total = 0.0
    with torch.no_grad():
        for imgs, masks in tqdm(loader, desc='Eval', leave=False):
            imgs, masks = imgs.to(device), masks.to(device)
            preds = model(imgs)
            if preds.shape != masks.shape:
                preds = nn.functional.interpolate(preds, size=masks.shape[-2:],
                                                  mode='bilinear', align_corners=False)
            dice_total += dice_score(preds, masks).item()
            iou_total  += iou_score(preds,  masks).item()
    return dice_total / len(loader), iou_total / len(loader)

print('Training functions ready ✓')

In [ ]:
# ── RUN TRAINING ─────────────────────────────────────────────────────────────
train_loader, val_loader, test_loader = get_loaders()

model     = get_model(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

best_iou    = 0.0
start_epoch = 1
ckpt_path   = os.path.join(CHECKPOINT_DIR, 'best_model.pth')

# Uncomment below to resume from a previous checkpoint you uploaded
# if os.path.exists(ckpt_path):
#     last = load_checkpoint(ckpt_path, model, optimizer)
#     start_epoch = last + 1
#     print(f'Resumed from epoch {last}')

print(f'Training from epoch {start_epoch} → {NUM_EPOCHS}')
print('-' * 60)

for epoch in range(start_epoch, NUM_EPOCHS + 1):
    train_loss        = train_one_epoch(model, train_loader, optimizer, DEVICE)
    val_dice, val_iou = evaluate(model, val_loader, DEVICE)
    scheduler.step()

    print(f'Epoch {epoch:02d}/{NUM_EPOCHS} | Loss: {train_loss:.4f} | '
          f'Val Dice: {val_dice:.4f} | Val IoU: {val_iou:.4f}')

    if val_iou > best_iou:
        best_iou = val_iou
        save_checkpoint(model, optimizer, epoch, ckpt_path)
        print(f'  → Best saved (IoU: {best_iou:.4f})')

print(f'\nTraining complete! Best Val IoU: {best_iou:.4f}')

In [ ]:
# ── TEST SET EVALUATION ───────────────────────────────────────────────────────
best_model = get_model(DEVICE)
load_checkpoint(ckpt_path, best_model)

test_dice, test_iou = evaluate(best_model, test_loader, DEVICE)

print('=' * 50)
print('  FINAL TEST SET RESULTS')
print(f'  Dice Score : {test_dice:.4f}')
print(f'  IoU  Score : {test_iou:.4f}')
print('=' * 50)

In [ ]:
# ── VISUALIZE PREDICTIONS ─────────────────────────────────────────────────────
NUM_SAMPLES = 6

vis_dataset = SOSDataset(TEST_IMG_DIR, TEST_MASK_DIR, get_transforms(False))
indices     = np.linspace(0, len(vis_dataset) - 1, NUM_SAMPLES, dtype=int)

fig, axes = plt.subplots(NUM_SAMPLES, 3, figsize=(14, NUM_SAMPLES * 4))
fig.suptitle('Oil Spill Detection — DeepLabV3+ + scSE | Sentinel-1 SAR',
             fontsize=14, fontweight='bold', y=1.01)

for col, title in enumerate(['SAR Input', 'Ground Truth (Red)', 'Prediction (Cyan)']):
    axes[0][col].set_title(title, fontsize=12, fontweight='bold')

best_model.eval()
for row, idx in enumerate(indices):
    image, mask = vis_dataset[idx]
    with torch.no_grad():
        logit = best_model(image.unsqueeze(0).to(DEVICE))
        prob  = torch.sigmoid(logit).squeeze().cpu().numpy()

    pred    = (prob > 0.5).astype(np.float32)
    mask_np = mask.squeeze().numpy()
    gray    = (image.permute(1, 2, 0).numpy() * 0.5 + 0.5).clip(0, 1)[:, :, 0]

    sample_iou  = iou_score(logit, mask.unsqueeze(0).to(DEVICE)).item()
    sample_dice = dice_score(logit, mask.unsqueeze(0).to(DEVICE)).item()

    axes[row][0].imshow(gray, cmap='gray')
    axes[row][0].set_ylabel(f'Sample {idx}\nIoU:{sample_iou:.3f} Dice:{sample_dice:.3f}',
                            fontsize=8, rotation=0, labelpad=90, va='center')

    axes[row][1].imshow(gray, cmap='gray')
    gt_ov = np.zeros((*mask_np.shape, 4))
    gt_ov[mask_np == 1] = [1, 0, 0, 0.55]
    axes[row][1].imshow(gt_ov)

    axes[row][2].imshow(gray, cmap='gray')
    pr_ov = np.zeros((*pred.shape, 4))
    pr_ov[pred == 1] = [0, 0.8, 1, 0.55]
    axes[row][2].imshow(pr_ov)

    for c in range(3): axes[row][c].axis('off')

fig.legend(handles=[
    mpatches.Patch(color=(1,0,0,0.7),   label='Ground Truth'),
    mpatches.Patch(color=(0,0.8,1,0.7), label='Prediction'),
], loc='lower center', ncol=2, fontsize=11, bbox_to_anchor=(0.5, -0.02))

plt.tight_layout()
save_path = os.path.join(PRED_DIR, 'predictions.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
print(f'Saved → {save_path}')
plt.show()